# Clean RaCA and DSP4SH for Google Earth Engine

This notebook performs source-data EDA, documents the joins and cleaning decisions, and creates two compact CSV files for Google Earth Engine scraping.

Outputs:

- `GEE_ready_datasets/raca_clean.csv`
- `GEE_ready_datasets/dsp4sh_clean.csv`
- `GEE_ready_datasets/raca_gee_points.csv` (one row per unique latitude/longitude)
- `GEE_ready_datasets/dsp4sh_gee_points.csv` (one row per unique latitude/longitude)

The raw Excel workbook and SQLite database are read-only inputs and are never modified. Rows are retained when they have a valid coordinate pair and a non-missing, non-negative SOC measurement. Measured bulk density is retained when available but is not imputed here.

In [ ]:
from pathlib import Path
from collections import Counter
import re
import sqlite3
import zipfile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# Works when run from the workspace root or from Final_Paper_Analysis/code.
cwd = Path.cwd().resolve()
if (cwd / "Final_Paper_Analysis" / "raw_data").exists():
    PROJECT_DIR = cwd / "Final_Paper_Analysis"
elif (cwd.parent / "raw_data").exists():
    PROJECT_DIR = cwd.parent
elif (cwd / "raw_data").exists():
    PROJECT_DIR = cwd
else:
    raise FileNotFoundError("Could not locate Final_Paper_Analysis/raw_data")

RAW_DIR = PROJECT_DIR / "raw_data"
OUT_DIR = PROJECT_DIR / "GEE_ready_datasets"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RACA_XLSX = RAW_DIR / "RaCA_2025.xlsx"
DSP_DB = RAW_DIR / "dsp4sh6.db"

print("Project:", PROJECT_DIR)
print("RaCA source:", RACA_XLSX)
print("DSP4SH source:", DSP_DB)

## Memory-efficient Excel reader

The principal RaCA sheet expands to roughly 244 MB of worksheet XML. Python's usual Excel readers can require excessive memory for this workbook, so the helper below uses R's compiled `readxl` reader to extract only the requested columns into a temporary directory. All subsequent EDA and cleaning remain in pandas. The source workbook is read-only and no intermediate extract is retained.

In [ ]:
import shutil
import subprocess
import tempfile

def load_raca_selected(xlsx_path, main_columns):
    if shutil.which("Rscript") is None:
        raise RuntimeError("Rscript is required to read this unusually large workbook efficiently")
    check = subprocess.run(
        ["Rscript", "-e", 'if (!requireNamespace("readxl", quietly=TRUE)) quit(status=2)'],
        capture_output=True, text=True,
    )
    if check.returncode != 0:
        raise RuntimeError("The R package readxl is required. Install it with install.packages('readxl').")

    with tempfile.TemporaryDirectory(prefix="raca_extract_") as tmp:
        main_csv = Path(tmp) / "raca_main.csv"
        xy_csv = Path(tmp) / "raca_xy.csv"
        r_script = Path(tmp) / "extract_raca.R"
        columns_r = "c(" + ",".join(repr(c) for c in main_columns) + ")"
        r_code = f'''
suppressPackageStartupMessages(library(readxl))
args <- commandArgs(trailingOnly=TRUE)
main_cols <- {columns_r}
main <- read_excel(args[[1]], sheet="RaCA_samples_wmodelBD")
missing <- setdiff(main_cols, names(main))
if (length(missing) > 0) stop(paste("Missing columns:", paste(missing, collapse=", ")))
write.csv(main[main_cols], args[[2]], row.names=FALSE, na="")
rm(main); gc()
xy <- read_excel(args[[1]], sheet="RaCA_xy")
write.csv(xy, args[[3]], row.names=FALSE, na="")
'''
        r_script.write_text(r_code)
        subprocess.run(
            ["Rscript", str(r_script), str(xlsx_path), str(main_csv), str(xy_csv)],
            check=True,
        )
        main = pd.read_csv(main_csv, low_memory=False)
        xy = pd.read_csv(xy_csv, low_memory=False)
    return main, xy

sheet_code = 'cat(readxl::excel_sheets(commandArgs(trailingOnly=TRUE)[1]), sep="\n")'
sheet_result = subprocess.run(
    ["Rscript", "-e", sheet_code, str(RACA_XLSX)], capture_output=True, text=True, check=True
)
print("Workbook sheets:")
print(sheet_result.stdout)

## 1. RaCA source EDA

The measurement table is horizon-level. Coordinates are held separately at the RaCA site level and are joined with `rcasiteid` → `RaCA_site`. The selected columns include every available site/sample/laboratory join identifier needed to trace an exported row back to its source.

In [ ]:
raca_columns = [
    "samp", "sample.id", "TOP", "BOT", "hzn_desgn", "rcasiteid", "pedon_no",
    "upedonid", "upedon", "Lab.Sample.No", "user_site_id", "smp_id",
    "natural_key", "lay_id", "c_tot_ncs", "caco3", "Measure_BD",
]
raca_raw, raca_xy = load_raca_selected(RACA_XLSX, raca_columns)

print("RaCA horizon table:", raca_raw.shape)
print("RaCA coordinate table:", raca_xy.shape)
display(raca_raw.head())
display(raca_xy.head())

raca_eda = pd.DataFrame({
    "dtype": raca_raw.dtypes.astype(str),
    "non_missing": raca_raw.notna().sum(),
    "missing": raca_raw.isna().sum(),
    "unique": raca_raw.nunique(dropna=True),
}).sort_values("missing", ascending=False)
display(raca_eda)

for col in ["c_tot_ncs", "caco3", "Measure_BD", "TOP", "BOT"]:
    raca_raw[col] = pd.to_numeric(raca_raw[col], errors="coerce")
display(raca_raw[["c_tot_ncs", "caco3", "Measure_BD", "TOP", "BOT"]].describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
raca_raw["c_tot_ncs"].dropna().clip(upper=raca_raw["c_tot_ncs"].quantile(0.99)).hist(bins=50, ax=axes[0])
axes[0].set(title="RaCA total carbon (through 99th percentile)", xlabel="c_tot_ncs (%)", ylabel="Rows")
raca_raw["Measure_BD"].dropna().hist(bins=50, ax=axes[1])
axes[1].set(title="RaCA measured bulk density", xlabel="g cm$^{-3}$", ylabel="Rows")
plt.tight_layout()
plt.show()

### RaCA cleaning rules

- Resolve the three duplicated `RaCA_site` coordinate keys by taking the mean coordinate; duplicates are identical or differ only at the sixth decimal place.
- Treat `(0, 0)` as a missing-location placeholder and require latitude/longitude within geographic bounds.
- Calculate SOC percent by mass using the established project rule: `c_tot_ncs - 0.12 × caco3` when positive carbonate is measured; otherwise use `c_tot_ncs`. Clip negative corrected values to zero.
- Retain only laboratory-measured bulk density (`Measure_BD`); do not substitute modelled bulk density.
- Require non-missing, non-negative SOC and a valid location. Missing measured bulk density is allowed because it is a separate response variable.
- Remove exact duplicate exported records, while preserving distinct horizons and samples.

In [ ]:
# One coordinate record per site. Mean safely resolves tiny rounding differences.
for col in ["Lat", "Lon"]:
    raca_xy[col] = pd.to_numeric(raca_xy[col], errors="coerce")
raca_xy_one = (
    raca_xy.groupby("RaCA_site", as_index=False)
    .agg({"Lat": "mean", "Lon": "mean", "Region": "first", "Landuse": "first", "Group": "first"})
)

raca = raca_raw.merge(
    raca_xy_one,
    how="left",
    left_on="rcasiteid",
    right_on="RaCA_site",
    validate="many_to_one",
)

positive_carbonate = raca["caco3"].notna() & (raca["caco3"] > 0)
raca["soc_pct_mass"] = raca["c_tot_ncs"]
raca.loc[positive_carbonate, "soc_pct_mass"] = (
    raca.loc[positive_carbonate, "c_tot_ncs"] - 0.12 * raca.loc[positive_carbonate, "caco3"]
)
raca["soc_pct_mass"] = raca["soc_pct_mass"].clip(lower=0)
raca["measured_bulk_density_g_cm3"] = pd.to_numeric(raca["Measure_BD"], errors="coerce")

valid_raca_coord = (
    raca["Lat"].between(-90, 90)
    & raca["Lon"].between(-180, 180)
    & ~((raca["Lat"] == 0) & (raca["Lon"] == 0))
)
valid_raca_soc = raca["soc_pct_mass"].notna() & (raca["soc_pct_mass"] >= 0)

raca_keep = [
    "Lat", "Lon", "rcasiteid", "samp", "sample.id", "pedon_no", "upedonid", "upedon",
    "Lab.Sample.No", "user_site_id", "smp_id", "natural_key", "lay_id",
    "hzn_desgn", "TOP", "BOT", "soc_pct_mass", "measured_bulk_density_g_cm3",
]
raca_clean = raca.loc[valid_raca_coord & valid_raca_soc, raca_keep].copy()
raca_clean = raca_clean.rename(columns={
    "Lat": "lat", "Lon": "lon", "sample.id": "sample_id",
    "Lab.Sample.No": "lab_sample_no", "TOP": "top_cm", "BOT": "bottom_cm",
})
raca_clean = raca_clean.drop_duplicates().reset_index(drop=True)

raca_qc = pd.Series({
    "source_horizon_rows": len(raca_raw),
    "source_unique_sites": raca_raw["rcasiteid"].nunique(),
    "coordinate_rows": len(raca_xy),
    "duplicate_coordinate_keys_resolved": int(raca_xy["RaCA_site"].duplicated(keep=False).sum()),
    "zero_zero_coordinate_sites_rejected": int(((raca_xy_one["Lat"] == 0) & (raca_xy_one["Lon"] == 0)).sum()),
    "rows_with_lab_carbon": int(raca["c_tot_ncs"].notna().sum()),
    "rows_with_measured_bulk_density": int(raca["Measure_BD"].notna().sum()),
    "export_rows": len(raca_clean),
    "export_unique_sites": raca_clean["rcasiteid"].nunique(),
    "export_rows_with_measured_bulk_density": int(raca_clean["measured_bulk_density_g_cm3"].notna().sum()),
}, name="count")
display(raca_qc.to_frame())
display(raca_clean.head())

## 2. DSP4SH database EDA and relational join

The DSP4SH export is assembled from:

- `cooplabmst`: sample IDs, SOC percent, and measured bulk density;
- `pedon`: pedon/plot IDs and longitude/latitude;
- `layerdesignation`: horizon designation and top/bottom depths.

The joins use the database's explicit keys: `DSP_Pedon_ID` for pedon coordinates and `DSP_sample_ID` for layer designation.

In [ ]:
con = sqlite3.connect(f"file:{DSP_DB}?mode=ro", uri=True)

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con
)["name"].tolist()
table_summary = []
for table in tables:
    n_rows = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM "{table}"', con).iloc[0, 0]
    n_cols = len(pd.read_sql_query(f'PRAGMA table_info("{table}")', con))
    table_summary.append({"table": table, "rows": n_rows, "columns": n_cols})
display(pd.DataFrame(table_summary))

for table in ["cooplabmst", "pedon", "layerdesignation"]:
    print(f"\n{table}")
    display(pd.read_sql_query(f'SELECT * FROM "{table}" LIMIT 5', con))

In [ ]:
dsp_query = '''
SELECT
    c.cooplabmst_ID,
    c.DSP_sample_ID,
    c.DSP_Pedon_ID,
    c.DSP_Pedon,
    c.layer_no,
    c.KSSL_labsampnum,
    c.SOC_pct,
    c.Bulk_Density,
    p.pedon_ID,
    p.DSP_Plot_ID,
    p.Pedon_Num,
    p.UserPedonID,
    p.pedon_x AS lon,
    p.pedon_y AS lat,
    l.laydesg_ID,
    l.hzdesg,
    l.hrzdep_t,
    l.hrzdep_b
FROM cooplabmst AS c
LEFT JOIN pedon AS p
    ON c.DSP_Pedon_ID = p.DSP_Pedon_ID
LEFT JOIN layerdesignation AS l
    ON c.DSP_sample_ID = l.DSP_sample_ID
'''
dsp = pd.read_sql_query(dsp_query, con)
con.close()

print("Joined DSP shape:", dsp.shape)
print("Unique laboratory rows:", dsp["cooplabmst_ID"].nunique())
assert len(dsp) == dsp["cooplabmst_ID"].nunique(), "DSP joins unexpectedly duplicated laboratory rows"

dsp_eda = pd.DataFrame({
    "dtype": dsp.dtypes.astype(str),
    "non_missing": dsp.notna().sum(),
    "missing": dsp.isna().sum(),
    "unique": dsp.nunique(dropna=True),
}).sort_values("missing", ascending=False)
display(dsp_eda)
display(dsp[["SOC_pct", "Bulk_Density", "lat", "lon", "hrzdep_t", "hrzdep_b"]].describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
dsp["SOC_pct"].dropna().clip(upper=dsp["SOC_pct"].quantile(0.99)).hist(bins=40, ax=axes[0])
axes[0].set(title="DSP4SH SOC (through 99th percentile)", xlabel="SOC (%)", ylabel="Rows")
dsp["Bulk_Density"].dropna().hist(bins=40, ax=axes[1])
axes[1].set(title="DSP4SH measured bulk density", xlabel="g cm$^{-3}$", ylabel="Rows")
plt.tight_layout()
plt.show()

### DSP4SH cleaning rules

- Require a valid, non-placeholder coordinate pair.
- Require non-missing, non-negative `SOC_pct`.
- Treat `Bulk_Density` as measured bulk density and retain missing values without imputation.
- Preserve every database ID and join key carried by the three joined source tables.
- Assert that the joins remain one output row per `cooplabmst_ID`, preventing accidental many-to-many duplication.

In [ ]:
numeric_dsp = ["lat", "lon", "SOC_pct", "Bulk_Density", "hrzdep_t", "hrzdep_b"]
for col in numeric_dsp:
    dsp[col] = pd.to_numeric(dsp[col], errors="coerce")

valid_dsp_coord = (
    dsp["lat"].between(-90, 90)
    & dsp["lon"].between(-180, 180)
    & ~((dsp["lat"] == 0) & (dsp["lon"] == 0))
)
valid_dsp_soc = dsp["SOC_pct"].notna() & (dsp["SOC_pct"] >= 0)

dsp_clean = dsp.loc[valid_dsp_coord & valid_dsp_soc].copy()
dsp_clean = dsp_clean.rename(columns={
    "SOC_pct": "soc_pct_mass",
    "Bulk_Density": "measured_bulk_density_g_cm3",
    "hrzdep_t": "top_cm",
    "hrzdep_b": "bottom_cm",
})
dsp_keep = [
    "lat", "lon", "cooplabmst_ID", "DSP_sample_ID", "DSP_Pedon_ID", "DSP_Pedon",
    "layer_no", "KSSL_labsampnum", "pedon_ID", "DSP_Plot_ID", "Pedon_Num",
    "UserPedonID", "laydesg_ID", "hzdesg", "top_cm", "bottom_cm",
    "soc_pct_mass", "measured_bulk_density_g_cm3",
]
dsp_clean = dsp_clean[dsp_keep].drop_duplicates().reset_index(drop=True)

dsp_qc = pd.Series({
    "source_lab_rows": dsp["cooplabmst_ID"].nunique(),
    "joined_rows": len(dsp),
    "rows_with_soc": int(dsp["SOC_pct"].notna().sum()),
    "rows_with_measured_bulk_density": int(dsp["Bulk_Density"].notna().sum()),
    "rows_with_horizon_designation": int(dsp["hzdesg"].notna().sum()),
    "export_rows": len(dsp_clean),
    "export_unique_pedons": dsp_clean["DSP_Pedon_ID"].nunique(),
    "export_rows_with_measured_bulk_density": int(dsp_clean["measured_bulk_density_g_cm3"].notna().sum()),
}, name="count")
display(dsp_qc.to_frame())
display(dsp_clean.head())

## 3. Final validation and export

CSV is used because it imports directly into Earth Engine as a table. Column names avoid spaces and periods. Identifier columns are exported as text where possible, while coordinates, depths, SOC, and bulk density remain numeric.

In [ ]:
def validate_gee_table(df, name, primary_id):
    assert len(df) > 0, f"{name} is empty"
    assert df["lat"].between(-90, 90).all(), f"{name}: invalid latitude"
    assert df["lon"].between(-180, 180).all(), f"{name}: invalid longitude"
    assert not ((df["lat"] == 0) & (df["lon"] == 0)).any(), f"{name}: placeholder coordinates remain"
    assert df["soc_pct_mass"].notna().all(), f"{name}: missing SOC remains"
    assert (df["soc_pct_mass"] >= 0).all(), f"{name}: negative SOC remains"
    assert df[primary_id].notna().all(), f"{name}: missing primary ID"
    assert not df.columns.str.contains(r"[ .]", regex=True).any(), f"{name}: GEE-unsafe column name"
    return {
        "dataset": name,
        "rows": len(df),
        "unique_primary_ids": df[primary_id].nunique(),
        "unique_sites_or_pedons": df["rcasiteid"].nunique() if "rcasiteid" in df else df["DSP_Pedon_ID"].nunique(),
        "soc_complete_pct": 100 * df["soc_pct_mass"].notna().mean(),
        "bulk_density_complete_pct": 100 * df["measured_bulk_density_g_cm3"].notna().mean(),
    }

def make_unique_gee_points(observations, prefix):
    points = (
        observations[["lat", "lon"]]
        .drop_duplicates()
        .sort_values(["lat", "lon"])
        .reset_index(drop=True)
    )
    points.insert(0, "gee_point_id", [f"{prefix}_{i:06d}" for i in range(1, len(points) + 1)])
    observations = observations.merge(points, on=["lat", "lon"], how="left", validate="many_to_one")
    assert observations["gee_point_id"].notna().all()
    assert not points[["lat", "lon"]].duplicated().any()
    return observations, points

raca_clean, raca_gee_points = make_unique_gee_points(raca_clean, "raca")
dsp_clean, dsp4sh_gee_points = make_unique_gee_points(dsp_clean, "dsp4sh")

validation = pd.DataFrame([
    validate_gee_table(raca_clean, "raca_clean", "samp"),
    validate_gee_table(dsp_clean, "dsp4sh_clean", "cooplabmst_ID"),
])
display(validation)

raca_path = OUT_DIR / "raca_clean.csv"
dsp_path = OUT_DIR / "dsp4sh_clean.csv"
raca_points_path = OUT_DIR / "raca_gee_points.csv"
dsp_points_path = OUT_DIR / "dsp4sh_gee_points.csv"
raca_clean.to_csv(raca_path, index=False)
dsp_clean.to_csv(dsp_path, index=False)
raca_gee_points.to_csv(raca_points_path, index=False)
dsp4sh_gee_points.to_csv(dsp_points_path, index=False)

print(f"Wrote {len(raca_clean):,} rows to {raca_path}")
print(f"Wrote {len(dsp_clean):,} rows to {dsp_path}")
print(f"Wrote {len(raca_gee_points):,} unique coordinate rows to {raca_points_path}")
print(f"Wrote {len(dsp4sh_gee_points):,} unique coordinate rows to {dsp_points_path}")
print("\nRaCA columns:", raca_clean.columns.tolist())
print("\nDSP4SH columns:", dsp_clean.columns.tolist())